# Fine-tuned XAI Robustness Study (GPU) — CSoNet 2026 companion

This notebook re-runs the plant-disease XAI robustness study with an **end-to-end fine-tuned** backbone
(features specialise to lesions, accuracy ~97-99%), a **larger evaluation set**, all **five CAM methods**,
and measures **faithfulness on both clean and perturbed inputs**. It saves CSVs/figures and prints
LaTeX-ready table rows to paste into the paper.

Runtime: **Runtime → Change runtime type → GPU (T4)**, then **Runtime → Run all**. ~30-50 min.

In [ ]:
# 1. Environment
import torch, os, sys, time
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!pip -q install grad-cam==1.5.5 datasets huggingface_hub scikit-image scikit-learn >/dev/null 2>&1
import numpy as np, json, random, collections, zipfile
import torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T, torchvision.transforms.functional as TF, torchvision.models as tvm
from PIL import Image
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"
ROOT="/content/pvxai"; os.makedirs(ROOT+"/data", exist_ok=True); os.makedirs(ROOT+"/results", exist_ok=True)
IMG=160; MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]

In [ ]:
# 2. Download PlantVillage (color) + leaf-based split, build a balanced subset
from huggingface_hub import hf_hub_download
N_TRAIN, N_TEST = 200, 60   # per class (GPU can afford more than the CPU study)
zip_path = hf_hub_download('mohanty/PlantVillage','data.zip',repo_type='dataset')
def load_split(name):
    p=hf_hub_download('mohanty/PlantVillage',f'splits/color_{name}.txt',repo_type='dataset')
    return [l.strip() for l in open(p) if l.strip()]
def by_class(lines):
    d=collections.defaultdict(list)
    for ln in lines: d[ln.split('/')[2]].append(ln)
    return d
tr,te=by_class(load_split('train')),by_class(load_split('test'))
classes=sorted(set(tr)|set(te)); cls2i={c:i for i,c in enumerate(classes)}
def sub(d,n):
    out={}
    for c,f in d.items():
        fs=sorted(f); random.Random(SEED).shuffle(fs); out[c]=fs[:min(n,len(fs))]
    return out
trs,tes=sub(tr,N_TRAIN),sub(te,N_TEST)
z=zipfile.ZipFile(zip_path); names=set(z.namelist())
DATA=ROOT+"/data/pv"
def extract(sel,split):
    items=[]
    for c,fs in sel.items():
        for m in fs:
            dst=os.path.join(DATA,split,c,os.path.basename(m))
            if not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst),exist_ok=True)
                with z.open(m) as s,open(dst,'wb') as o: o.write(s.read())
            items.append((dst,cls2i[c]))
    return items
train_items=extract(trs,'train'); test_items=extract(tes,'test')
print("classes",len(classes),"train",len(train_items),"test",len(test_items))

In [ ]:
# 3. Datasets / loaders
norm=T.Normalize(MEAN,STD); rsz=T.Resize((IMG,IMG))
class DS(torch.utils.data.Dataset):
    def __init__(self,items,train=False): self.items=items; self.train=train
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,y=self.items[i]; im=rsz(Image.open(p).convert('RGB'))
        if self.train and random.random()<0.5: im=TF.hflip(im)
        raw=T.ToTensor()(im); return norm(raw.clone()), y
tl=torch.utils.data.DataLoader(DS(train_items,True),batch_size=64,shuffle=True,num_workers=2)
vl=torch.utils.data.DataLoader(DS(test_items),batch_size=128,num_workers=2)

In [ ]:
# 4. END-TO-END FINE-TUNING (EfficientNet-B0). Set also_resnet=True to add ResNet-50.
from sklearn.metrics import f1_score, accuracy_score
def build(name):
    if name=='efficientnet_b0':
        m=tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1]=nn.Linear(m.classifier[1].in_features,len(classes)); tgt='features'
    elif name=='resnet50':
        m=tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        m.fc=nn.Linear(m.fc.in_features,len(classes)); tgt='layer4'
    return m.to(DEV)
def finetune(name,epochs=6,lr=3e-4):
    m=build(name); opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs); lossf=nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train(); t0=time.time()
        for x,y in tl:
            x,y=x.to(DEV),y.to(DEV); opt.zero_grad(); loss=lossf(m(x),y); loss.backward(); opt.step()
        sched.step()
        m.eval(); P=[]; Y=[]
        with torch.no_grad():
            for x,y in vl: P+= m(x.to(DEV)).argmax(1).cpu().tolist(); Y+=y.tolist()
        print(f"{name} epoch {ep+1}/{epochs} acc={accuracy_score(Y,P):.4f} f1={f1_score(Y,P,average='macro'):.4f} ({time.time()-t0:.0f}s)")
    torch.save(m.state_dict(), f"{ROOT}/results/{name}_ft.pt")
    return m, accuracy_score(Y,P), f1_score(Y,P,average='macro')
also_resnet=False
models={}
m_eff,acc_eff,f1_eff=finetune('efficientnet_b0'); models['efficientnet_b0']=m_eff
print("EfficientNet-B0 fine-tuned:",acc_eff,f1_eff)
if also_resnet:
    m_res,acc_res,f1_res=finetune('resnet50'); models['resnet50']=m_res

In [ ]:
# 5. Transform protocol + metrics (identical definitions to the CPU study) + faithfulness under perturbation
from skimage.metrics import structural_similarity as ssim_fn
ROT=15; BLUR=2.0; BRIGHT=1.3; NOISE=0.08; TRANSFORMS=['blur','rotate','brightness','noise']
def _n(raw): return TF.normalize(raw.clamp(0,1),MEAN,STD)
def apply_t(raw,name):
    if name=='identity': r=raw.clone()
    elif name=='blur': r=TF.gaussian_blur(raw,9,BLUR)
    elif name=='rotate': r=TF.rotate(raw,ROT,interpolation=TF.InterpolationMode.BILINEAR)
    elif name=='brightness': r=(raw*BRIGHT).clamp(0,1)
    elif name=='noise': r=(raw+torch.randn_like(raw)*NOISE).clamp(0,1)
    return _n(r).unsqueeze(0).to(DEV), r
def align(cam,name):
    H,W=cam.shape
    if name=='rotate':
        t=torch.tensor(cam).unsqueeze(0)
        rc=TF.rotate(t,ROT,interpolation=TF.InterpolationMode.BILINEAR)[0].numpy()
        vm=TF.rotate(torch.ones(1,H,W),ROT,interpolation=TF.InterpolationMode.NEAREST)[0].numpy()>0.5
        return rc,vm
    return cam.copy(), np.ones((H,W),bool)
def pear(a,b,m):
    x,y=a[m].ravel(),b[m].ravel()
    return 0.0 if x.std()<1e-8 or y.std()<1e-8 else float(np.corrcoef(x,y)[0,1])
def ssm(a,b,m):
    aa,bb=a.copy(),b.copy(); aa[~m]=0; bb[~m]=0; return float(ssim_fn(aa,bb,data_range=1.0))
def iou(a,b,m,k=0.2):
    va,vb=a[m],b[m]
    if va.size==0: return 0.0
    ba=(a>=np.quantile(va,1-k))&m; bb=(b>=np.quantile(vb,1-k))&m
    u=np.logical_or(ba,bb).sum(); return float(np.logical_and(ba,bb).sum()/u) if u else 0.0
def prob(m,x,c):
    with torch.no_grad(): return float(F.softmax(m(x),1)[0,c])
def delins(m,raw,cam,c,steps=20):
    H,W=cam.shape; order=np.argsort(-cam.ravel()); n=H*W; step=max(1,n//steps)
    base=TF.gaussian_blur(raw,15,8.0); di=raw.clone().reshape(3,-1); ii=base.clone().reshape(3,-1); fr=raw.reshape(3,-1)
    dp=[prob(m,_n(raw).unsqueeze(0).to(DEV),c)]; ip=[prob(m,_n(base).unsqueeze(0).to(DEV),c)]
    for s in range(0,n,step):
        idx=order[s:s+step]; di[:,idx]=0; ii[:,idx]=fr[:,idx]
        dp.append(prob(m,_n(di.reshape(3,H,W)).unsqueeze(0).to(DEV),c))
        ip.append(prob(m,_n(ii.reshape(3,H,W)).unsqueeze(0).to(DEV),c))
    return float(np.trapz(dp,dx=1/(len(dp)-1))), float(np.trapz(ip,dx=1/(len(ip)-1)))
def avgdrop(m,raw,cam,c):
    pf=prob(m,_n(raw).unsqueeze(0).to(DEV),c)
    pm=prob(m,_n((raw*torch.tensor(cam)).clamp(0,1)).unsqueeze(0).to(DEV),c)
    return float(max(0,pf-pm)/(pf+1e-8)*100)

In [ ]:
# 6. XAI evaluation on a larger correctly-classified set, all five methods.
#    Faithfulness measured on BOTH clean and each perturbed input (#6).
from pytorch_grad_cam import GradCAM,GradCAMPlusPlus,ScoreCAM,EigenCAM,LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
CAMS={'GradCAM':GradCAM,'GradCAM++':GradCAMPlusPlus,'ScoreCAM':ScoreCAM,'EigenCAM':EigenCAM,'LayerCAM':LayerCAM}
def tgt_layer(m,name): return [m.features[-1]] if name=='efficientnet_b0' else [m.layer4[-1]]
N_XAI=100   # correctly-classified images for the explanation study (GPU: all methods)
def load_raw(p): return T.ToTensor()(rsz(Image.open(p).convert('RGB')))
records=[]
for name,model in models.items():
    model.eval()
    # pick N_XAI correctly-classified test images (balanced-ish by iteration order)
    picked=[]
    for p,y in test_items:
        raw=load_raw(p); x=_n(raw).unsqueeze(0).to(DEV)
        with torch.no_grad(): c=int(model(x).argmax(1))
        if c==y: picked.append((p,y))
        if len(picked)>=N_XAI: break
    print(name,"xai images:",len(picked))
    tl_=tgt_layer(model,name)
    for gi,(p,y) in enumerate(picked):
        raw=load_raw(p); x=_n(raw).unsqueeze(0).to(DEV)
        with torch.no_grad(): c=int(model(x).argmax(1))
        tgt=[ClassifierOutputTarget(c)]
        for mname,CAM in CAMS.items():
            with CAM(model=model,target_layers=tl_) as cam:
                if mname=='ScoreCAM': cam.batch_size=256
                cam0=cam(input_tensor=x,targets=tgt)[0]
            dA,iA=delins(model,raw,cam0,c); ad=avgdrop(model,raw,cam0,c)
            rec={'backbone':name,'method':mname,'img':gi,'del_clean':dA,'ins_clean':iA,'ad_clean':ad,'stab':{},'faith_pert':{}}
            for t in TRANSFORMS:
                xt,rawt=apply_t(raw,t)
                with CAM(model=model,target_layers=tl_) as cam:
                    if mname=='ScoreCAM': cam.batch_size=256
                    camt=cam(input_tensor=xt,targets=tgt)[0]
                al,vm=align(cam0,t)
                rec['stab'][t]={'pearson':pear(al,camt,vm),'ssim':ssm(al,camt,vm),'iou':iou(al,camt,vm)}
                # faithfulness under perturbation (#6): del/ins on the perturbed image w/ its own map
                dP,iP=delins(model,rawt,camt,c)
                rec['faith_pert'][t]={'del':dP,'ins':iP}
            records.append(rec)
    print(f"  done {name}: {len(records)} records so far")
json.dump(records, open(f"{ROOT}/results/ft_xai_records.json","w"))
print("saved", len(records), "records")

In [ ]:
# 7. Aggregate: bootstrap CIs, per-transform + pooled Holm, clean vs perturbed faithfulness. Emit LaTeX rows.
import itertools, pandas as pd
from scipy.stats import wilcoxon
rng=np.random.default_rng(SEED); df=pd.DataFrame(records)
def boot(v):
    v=np.asarray(v,float); b=[rng.choice(v,len(v),True).mean() for _ in range(2000)]
    return v.mean(),np.percentile(b,2.5),np.percentile(b,97.5)
def pim(g,met): return np.array([np.mean([r['stab'][t][met] for t in TRANSFORMS]) for r in g.to_dict('records')])
print("=== Stability (mean [95% CI]) — FINE-TUNED ===")
for (bb,mm),g in df.groupby(['backbone','method']):
    pe=boot(pim(g,'pearson')); print(f"{bb} & {mm} & {pe[0]:.3f} [{pe[1]:.3f},{pe[2]:.3f}] \\\\")
print("\n=== Faithfulness: clean vs perturbed (avg over transforms) ===")
for (bb,mm),g in df.groupby(['backbone','method']):
    dc=boot(g['del_clean'].values); ic=boot(g['ins_clean'].values)
    dp=np.mean([np.mean([r['faith_pert'][t]['del'] for t in TRANSFORMS]) for r in g.to_dict('records')])
    ip=np.mean([np.mean([r['faith_pert'][t]['ins'] for t in TRANSFORMS]) for r in g.to_dict('records')])
    print(f"{bb} {mm}: del clean={dc[0]:.3f} pert={dp:.3f} | ins clean={ic[0]:.3f} pert={ip:.3f}")
print("\n=== Pooled Holm (per-image mean Pearson) ===")
for bb in df.backbone.unique():
    ms=sorted(df[df.backbone==bb].method.unique()); per={m:pim(df[(df.backbone==bb)&(df.method==m)],'pearson') for m in ms}
    ps=[]
    for a,b in itertools.combinations(ms,2):
        try: _,p=wilcoxon(per[a],per[b])
        except ValueError: p=1.0
        ps.append([a,b,float(per[a].mean()-per[b].mean()),p])
    order=np.argsort([x[3] for x in ps]); mt=len(ps); adj=[0]*mt; run=0
    for r,i in enumerate(order): run=max(run,(mt-r)*ps[i][3]); adj[i]=min(1,run)
    for i,(a,b,d,p) in enumerate(ps):
        if adj[i]<0.05: print(f"  {bb}: {a} vs {b} diff={d:+.3f} p_holm={adj[i]:.4f}")

In [ ]:
# 8. Save figures + zip everything for download
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
COL={'GradCAM':'#1f77b4','GradCAM++':'#ff7f0e','ScoreCAM':'#2ca02c','EigenCAM':'#d62728','LayerCAM':'#9467bd'}
fig,ax=plt.subplots(figsize=(7,4))
for (bb,mm),g in df.groupby(['backbone','method']):
    xs=[np.mean([r['stab'][t]['pearson'] for r in g.to_dict('records')]) for t in TRANSFORMS]
    ax.plot(TRANSFORMS,xs,'-o',label=f"{mm}",color=COL.get(mm))
ax.set_ylabel('Explanation stability (Pearson r)'); ax.set_title('Fine-tuned EfficientNet-B0'); ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{ROOT}/results/ft_stability.png",dpi=150)
import shutil; shutil.make_archive('/content/ft_results','zip',f"{ROOT}/results")
from google.colab import files
print("Download the results bundle:")
files.download('/content/ft_results.zip')